In [4]:
!pip install \
  torch_geometric \
  pyg_lib torch_scatter \
  torch_sparse \
  torch_cluster \
  torch_spline_conv \
  -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!pip install imbalanced-learn \
  tqdm

Looking in links: https://data.pyg.org/whl/torch-2.6.0+cu124.html


In [5]:
"""
03_Model_Training.py (Optimized Version)
–––––––––––––––––––––––––––––––––––––––––––––––––––––––––––––
• trains CyberThreatDetector with optimal hyperparameters from seed 50
• picks the decision threshold on the validation set
• evaluates once on the test set with that fixed threshold
• saves best FP32 checkpoint
"""

#0. Imports & Seed
import os, time, pickle, random, numpy as np, torch
import torch.nn.functional as F
from torch.optim       import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.nn.utils import clip_grad_norm_
from torch_geometric.data   import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn     import SAGEConv, BatchNorm
from sklearn.metrics import f1_score, precision_score, recall_score
from google.colab import drive
from tqdm.auto import tqdm

def set_seed(seed: int = 50):  # Changed to best seed (50)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
# -------------------------------------------------------------

#1. Paths & Drive
drive.mount('/content/drive', force_remount=True)

DRIVE_PATH = '/content/drive/MyDrive/CyberThreatDetectionSystem_Project/'
DATA_PATH  = os.path.join(DRIVE_PATH, 'Data/processed/')
MODEL_PATH = os.path.join(DRIVE_PATH, 'Models/')
os.makedirs(MODEL_PATH, exist_ok=True)

#2. Data helpers
def load_graph_data():
    def _load(split):
        with open(os.path.join(DATA_PATH, f'{split}_graph.pkl'), 'rb') as f:
            return pickle.load(f)
    return _load('train'), _load('val'), _load('test')

def process_graph_data(raw_train, raw_val, raw_test):
    # Use robust normalization with quantiles
    train_feats = torch.tensor(raw_train['x'], dtype=torch.float32)
    q_low, q_high = torch.quantile(train_feats, torch.tensor([0.01, 0.99]), dim=0)
    iqr = torch.where(q_high - q_low > 1e-6, q_high - q_low, torch.ones_like(q_low))

    def _to_data(obj):
        x = torch.tensor(obj['x'], dtype=torch.float32)
        x_scaled = torch.clamp((x - q_low) / iqr, -5.0, 5.0)
        return Data(x=x_scaled,
                    edge_index=torch.tensor(obj['edge_index'], dtype=torch.long),
                    y=torch.tensor(obj['y'], dtype=torch.long))
    return _to_data(raw_train), _to_data(raw_val), _to_data(raw_test)

def create_loaders(train, val, test, batch_size=4096):
    # Optimal neighbor sizes from the successful model
    num_neighbors = [50, 40, 30, 20]
    return (
        NeighborLoader(train, num_neighbors=num_neighbors, batch_size=batch_size, shuffle=True),
        NeighborLoader(val,   num_neighbors=num_neighbors, batch_size=batch_size, shuffle=False),
        NeighborLoader(test,  num_neighbors=num_neighbors, batch_size=batch_size, shuffle=False)
    )

#3. Model architecture
class CyberThreatDetector(torch.nn.Module):
    def __init__(self, in_channels, hidden=192, dropout=0.20, n_layers=4):  # Updated to optimal values
        super().__init__()
        # Input projection with layer normalization
        self.input_proj = torch.nn.Linear(in_channels, hidden)
        self.input_norm = torch.nn.LayerNorm(hidden)

        # GraphSAGE layers with residual connections and layer normalization
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.lns = torch.nn.ModuleList()

        for _ in range(n_layers):
            self.convs.append(SAGEConv(hidden, hidden, aggr="mean"))
            self.bns.append(BatchNorm(hidden))
            self.lns.append(torch.nn.LayerNorm(hidden))

        # Additional attention layer for better feature extraction
        self.attention = torch.nn.MultiheadAttention(
            embed_dim=hidden,
            num_heads=4,
            dropout=dropout,
            batch_first=True
        )

        # Output projection with two layers for better expressivity
        self.fc1 = torch.nn.Linear(hidden, hidden // 2)
        self.output = torch.nn.Linear(hidden // 2, 1)

        self.dropout = torch.nn.Dropout(dropout)
        self.gelu = torch.nn.GELU()
        self.n_layers = n_layers

    def forward(self, x, edge_index):
        # Project input features to hidden dimension with normalization
        x = self.input_proj(x)
        x = self.input_norm(x)
        x = self.gelu(x)

        # Process through GNN layers
        for i in range(self.n_layers):
            identity = x
            x = self.lns[i](x)
            x = self.convs[i](x, edge_index)
            x = self.bns[i](x)
            x = self.gelu(x)
            x = self.dropout(x)
            x = x + identity  # Residual connection

        # Apply self-attention for better feature interaction
        # Reshape for attention layer input requirements
        x_reshaped = x.unsqueeze(1)  # [batch_size, 1, hidden]
        x_attn, _ = self.attention(x_reshaped, x_reshaped, x_reshaped)
        x = x + x_attn.squeeze(1)  # Residual connection with attention output

        # Final output projection
        x = self.fc1(x)
        x = self.gelu(x)
        x = self.dropout(x)
        return self.output(x).squeeze()

#4. Evaluation utilities
@torch.no_grad()
def collect_probs(model, loader, device):
    model.eval()
    probs, labels = [], []
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch.x, batch.edge_index)[:batch.batch_size]
        probs.append(torch.sigmoid(logits).cpu())
        labels.append(batch.y[:batch.batch_size].cpu())
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()

def best_threshold(model, val_loader, device='cpu', lo=0.05, hi=0.30, steps=5000):
    probs, labels = collect_probs(model, val_loader, device)
    ts = np.linspace(lo, hi, steps)
    f1s = [f1_score(labels, probs > t) for t in ts]
    idx = int(np.argmax(f1s))
    return ts[idx], f1s[idx]

def evaluate_fixed(model, loader, thr, device='cpu'):
    probs, labels = collect_probs(model, loader, device)
    preds = probs > thr
    return {
        'f1':        f1_score(labels, preds),
        'precision': precision_score(labels, preds),
        'recall':    recall_score(labels, preds),
        'threshold': thr
    }

#5. Training routine
def train_model(epochs=200, batch_size=4096, lr=0.007,
                weight_decay=3.5e-5, early_stop=50, target_threshold=0.0596):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    n_normal = (train_data.y == 0).sum()
    n_attack = (train_data.y == 1).sum()
    pos_weight = torch.tensor([1.33 * n_normal / n_attack], dtype=torch.float32).to(device)
    print(f"Class balance – Normal: {n_normal}, Attack: {n_attack}")

    model = CyberThreatDetector(
        in_channels=train_data.x.size(1),
        hidden=192,
        dropout=0.20,
        n_layers=4
    ).to(device)

    optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.75,
                                  patience=6, verbose=False)

    best_f1, patience = 0, 0
    best_path = os.path.join(MODEL_PATH, 'model_log_example.pth')

    print("\n===== Training =====")
    for epoch in tqdm(range(1, epochs + 1), unit="epoch"):
        model.train()
        total_loss = 0

        # Learning rate warmup for first 5 epochs
        if epoch < 5:
            warmup_factor = (epoch + 1) / 5
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr * warmup_factor

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            logits = model(batch.x, batch.edge_index)[:batch.batch_size]
            targets = batch.y[:batch.batch_size].float()

            # Apply label smoothing
            smoothed = targets * (1 - 0.03) + 0.01  # 0.03 is label_smoothing param, adding 0.01

            loss = F.binary_cross_entropy_with_logits(logits, smoothed,
                                                      pos_weight=pos_weight)
            loss.backward()

            # Gradient clipping
            clip_grad_norm_(model.parameters(), max_norm=2.5)  # Updated

            optimizer.step()
            total_loss += loss.item()

        # Validation check
        val_metrics = evaluate_fixed(model, val_loader, target_threshold, device)

        print(f"Epoch {epoch}: loss={total_loss:.4f} — "
          f"Val F1={val_metrics['f1']:.4f}@thr={val_metrics['threshold']:.3f}")

        # Only adjust LR if not in warmup phase
        if epoch >= 5:
            scheduler.step(val_metrics['f1'])

        if val_metrics['f1'] > best_f1:
            best_f1 = val_metrics['f1']
            patience = 0
            torch.save({'model_state_dict': model.state_dict()}, best_path)
        else:
            patience += 1
            if patience >= early_stop:
                print(f"Early stopping at epoch {epoch}")
                break

    # Load best model
    model.load_state_dict(torch.load(best_path)['model_state_dict'])
    return model

#6. Main execution
if __name__ == "__main__":
    set_seed(50)  # Using the best seed

    # data
    raw_train, raw_val, raw_test = load_graph_data()
    train_data, val_data, test_data = process_graph_data(raw_train, raw_val, raw_test)
    train_loader, val_loader, test_loader = create_loaders(
        train_data, val_data, test_data, batch_size=4096
    )

    # train
    best_model = train_model()

    # first evaluate with the optimal threshold from ensemble study ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    optimal_test_metrics = evaluate_fixed(best_model, test_loader, 0.0596, device)
    print("\n===== TEST METRICS WITH OPTIMAL THRESHOLD =====")
    for k in ('f1', 'precision', 'recall'):
        print(f"{k.capitalize():<9}: {optimal_test_metrics[k]:.6f}")
    print(f"Threshold   : 0.0596 (from best seed model)")

    # choose threshold on validation only (for comparison) ---
    best_model.to('cpu')
    best_thr, best_val_f1 = best_threshold(best_model, val_loader, device='cpu')
    print(f"\nValidation-chosen threshold: {best_thr:.6f}  |  Val F-1 = {best_val_f1:.6f}")

    # evaluate once on test with validation-chosen threshold ---
    test_metrics = evaluate_fixed(best_model, test_loader, best_thr)
    print("\n===== FINAL TEST METRICS (VAL-CHOSEN THRESHOLD) =====")
    for k in ('f1', 'precision', 'recall'):
        print(f"{k.capitalize():<9}: {test_metrics[k]:.6f}")
    print(f"Threshold   : {best_thr:.6f}")

Mounted at /content/drive
Using device: cuda
Class balance – Normal: 7012, Attack: 46435

===== Training =====


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


  0%|          | 0/200 [00:00<?, ?epoch/s]

Epoch 1: loss=1.8044 — Val F1=0.9266@thr=0.060
Epoch 2: loss=1.2272 — Val F1=0.9827@thr=0.060
Epoch 3: loss=1.1665 — Val F1=0.9778@thr=0.060
Epoch 4: loss=1.1347 — Val F1=0.9692@thr=0.060
Epoch 5: loss=1.0672 — Val F1=0.9886@thr=0.060
Epoch 6: loss=1.0443 — Val F1=0.9880@thr=0.060
Epoch 7: loss=1.0452 — Val F1=0.9894@thr=0.060
Epoch 8: loss=1.0172 — Val F1=0.9908@thr=0.060
Epoch 9: loss=0.9954 — Val F1=0.9909@thr=0.060
Epoch 10: loss=1.0018 — Val F1=0.9895@thr=0.060
Epoch 11: loss=1.0063 — Val F1=0.9841@thr=0.060
Epoch 12: loss=1.0516 — Val F1=0.9852@thr=0.060
Epoch 13: loss=1.0524 — Val F1=0.9892@thr=0.060
Epoch 14: loss=0.9910 — Val F1=0.9836@thr=0.060
Epoch 15: loss=1.0807 — Val F1=0.9918@thr=0.060
Epoch 16: loss=1.0006 — Val F1=0.9912@thr=0.060
Epoch 17: loss=0.9652 — Val F1=0.9913@thr=0.060
Epoch 18: loss=0.9552 — Val F1=0.9919@thr=0.060
Epoch 19: loss=0.9745 — Val F1=0.9924@thr=0.060
Epoch 20: loss=0.9775 — Val F1=0.9877@thr=0.060
Epoch 21: loss=0.9603 — Val F1=0.9889@thr=0.060
E